# Simulation

### Libraries

In [ ]:
import pandas as pd
import numpy as np
import os

from datetime import timedelta
import plotly.express as px
from scipy import stats

os.chdir("../2 - SIMULATION")

### Import and Clean Data

In [2]:
rental_df = pd.read_csv('rental_data.csv')
rental_df['Start'] = pd.to_datetime(rental_df['Start'])
rental_df['End'] = pd.to_datetime(rental_df['End'])
rental_df.head(10)

,Rental_state,Start,End,Revenue,Distance
0,RED,2017-04-18 20:13:45.279,2017-04-18 20:30:13.041,4.43,8
1,GREEN,2017-09-17 19:06:00.982,2017-09-17 19:07:15.896,0.00,0
2,RED,2017-07-28 21:56:11.499,2017-07-28 22:06:50.314,2.87,3
3,GREEN,2017-01-31 21:31:57.878,2017-01-31 21:50:07.872,0.00,0
4,RED,2017-08-09 20:34:39.686,2017-08-09 20:49:32.039,2.82,5
5,RED,2017-07-01 18:09:06.723,2017-07-01 18:31:02.322,4.23,7
6,GREEN,2017-11-28 14:11:36.965,2017-11-28 14:39:15.297,0.00,0
7,GREEN,2017-08-01 10:00:22.398,2017-08-01 10:11:22.661,0.00,0
8,GREEN,2017-01-16 16:21:21.886,2017-01-16 16:42:33.288,0.00,0
9,RED,2017-09-15 19:44:44.400,2017-09-15 19:58:07.481,3.06,4


In [3]:
charge_prob = pd.read_csv('charge_probabilites.csv', sep=';')
charge_prob["Probability_Percent"] = charge_prob["Charging_probability"] / 100
charge_prob.head(10)

,Incentive,Charging_probability,Probability_Percent
0,0,0,0.00
1,5,8,0.08
2,10,16,0.16
3,15,24,0.24
4,20,32,0.32
5,25,40,0.40
6,30,48,0.48
7,35,56,0.56
8,40,64,0.64
9,45,72,0.72


## Calculations

Assumptions:

1) The first appearance of a car is at 100% battery life.
2) Probabilities given are in percentages, so dividing by 100 gives the raw probability.
3) Rental duration is Start time subtracted from End time.
4) Revenue per minute is Revenue divided by Revenue collected from customer.
5) Car battery has 100% efficiency. 
6) If battery consumption is 0.5% per km then a car with a battery of 100% efficiency can drive for 200 km. 
7) If the trip has lasted longer than 200km then the customer charged the car. 
8) Data has been sorted in chronological order to enable assingment of car id variable.
9) A car first appears in the dataset with a rental state of GREEN. 
10) Car state is switched to RED when battery falls below 20%, i.e. after 160km. 
11) If one row has a distance than exceeds 160km, we assume that the customer charged the car. 
12) Customer always parks their car at a charging station at the end of a trip when in RED state. 
13) Car recharge time is 6 hours. Minimum time to get back on road = 180 + (6 * 60) = 540 minutes
14) There is a delay of MINIMUM 180 minutes + 6 hours recharge time before a car in RED state is allowed back on the road i.e. GREEN
15) No car overlap. If the end time for GREEN state is greater than the next start time, then this is another car with a different id. 
16) No maintenance is done other than charging the car for the duration of the dataset. 
17) If State is RED but Revenue and Distance are 0, then we assume is this a cancelled ride. 
18) If the State is GREEN but no Revenue, then this was a car put up for rent but unchosen.  

In [4]:
cars_df = rental_df.copy()
cars_df['duration_minute'] = (cars_df['End'] - cars_df['Start']).dt.total_seconds() / 60
cars_df['revenue_minute'] = cars_df['Revenue'] / cars_df['duration_minute']
cars_df['revenue_km'] = (cars_df['Revenue'] / cars_df['Distance']).fillna(0)
cars_df.head(10)

,Rental_state,Start,End,Revenue,Distance,duration_minute,revenue_minute,revenue_km
0,RED,2017-04-18 20:13:45.279,2017-04-18 20:30:13.041,4.43,8,16.462700,0.269093,0.553750
1,GREEN,2017-09-17 19:06:00.982,2017-09-17 19:07:15.896,0.00,0,1.248567,0.000000,0.000000
2,RED,2017-07-28 21:56:11.499,2017-07-28 22:06:50.314,2.87,3,10.646917,0.269562,0.956667
3,GREEN,2017-01-31 21:31:57.878,2017-01-31 21:50:07.872,0.00,0,18.166567,0.000000,0.000000
4,RED,2017-08-09 20:34:39.686,2017-08-09 20:49:32.039,2.82,5,14.872550,0.189611,0.564000
5,RED,2017-07-01 18:09:06.723,2017-07-01 18:31:02.322,4.23,7,21.926650,0.192916,0.604286
6,GREEN,2017-11-28 14:11:36.965,2017-11-28 14:39:15.297,0.00,0,27.638867,0.000000,0.000000
7,GREEN,2017-08-01 10:00:22.398,2017-08-01 10:11:22.661,0.00,0,11.004383,0.000000,0.000000
8,GREEN,2017-01-16 16:21:21.886,2017-01-16 16:42:33.288,0.00,0,21.190033,0.000000,0.000000
9,RED,2017-09-15 19:44:44.400,2017-09-15 19:58:07.481,3.06,4,13.384683,0.228620,0.765000


In [5]:
cars_df.loc[cars_df['Distance'].idxmax()]

Rental_state                              RED
Start              2017-05-19 07:51:09.300000
End                2017-05-20 04:29:45.216000
Revenue                                217.35
Distance                                  716
duration_minute                     1238.5986
revenue_minute                       0.175481
revenue_km                           0.303561
Name: 97540, dtype: object

In [6]:
#Sort data in chronological order
cars_df_sorted = cars_df.sort_values(by='Start', ascending=True)
cars_df_sorted.head(10)

,Rental_state,Start,End,Revenue,Distance,duration_minute,revenue_minute,revenue_km
72265,GREEN,2017-01-01 00:13:10.027,2017-01-01 00:17:56.452,0.00,0,4.773750,0.000000,0.000000
69011,RED,2017-01-01 00:19:04.324,2017-01-01 00:39:20.440,4.57,4,20.268600,0.225472,1.142500
30272,GREEN,2017-01-01 00:20:57.892,2017-01-01 02:48:11.075,0.00,0,147.219717,0.000000,0.000000
78025,RED,2017-01-01 00:21:40.974,2017-01-01 00:45:19.135,4.84,19,23.636017,0.204772,0.254737
8757,GREEN,2017-01-01 00:37:10.018,2017-01-01 00:41:21.268,0.00,0,4.187500,0.000000,0.000000
61048,RED,2017-01-01 00:37:11.831,2017-01-01 01:18:03.889,10.33,16,40.867633,0.252767,0.645625
76277,RED,2017-01-01 00:37:52.707,2017-01-01 00:50:03.954,2.42,7,12.187450,0.198565,0.345714
47985,RED,2017-01-01 00:41:27.955,2017-01-01 00:42:43.816,0.00,0,1.264350,0.000000,0.000000
83252,RED,2017-01-01 00:45:17.930,2017-01-01 01:06:59.664,4.44,7,21.695567,0.204650,0.634286
39944,RED,2017-01-01 00:47:59.854,2017-01-01 01:00:55.013,2.62,4,12.919317,0.202797,0.655000


### Assign car id to each car

In [ ]:
car_data = {}          
available_cars = []    
car_counter = 1

car_ids = []
accum_distances = []

for idx, row in cars_df_sorted.iterrows():
    start, end = row['Start'], row['End']
    distance, state, revenue = row['Distance'], row['Rental_state'], row['Revenue']
    assigned = False

    for car_id in available_cars:
        info = car_data[car_id]
        next_free_time = max(info['last_end'], info.get('cooldown_until', start))

        #Check availability, ensuring no consecutive GREEN states
        if start >= next_free_time and not (state == "GREEN" and info.get('last_state') == "GREEN"):
            car_ids.append(car_id)

            #Calculate new accumulated distance
            new_accum = info.get('accum_distance', 0) + distance

            #Reset only if distance >= 160 and current row is not zero-revenue/zero-distance
            if new_accum >= 160 and not (distance == 0 and revenue == 0):
                info['accum_distance'] = distance
                info['cooldown_until'] = end + timedelta(minutes=540)
            else:
                info['accum_distance'] = new_accum
                info['cooldown_until'] = start

            info['last_end'] = end
            info['last_state'] = state

            accum_distances.append(info['accum_distance'])
            assigned = True
            break

    if not assigned:
        car_ids.append(car_counter)
        car_data[car_counter] = {
            'last_end': end,
            'accum_distance': distance,
            'cooldown_until': end + timedelta(minutes=540) if distance >= 160 and not (distance == 0 and revenue == 0) else start,
            'last_state': state
        }
        accum_distances.append(distance)
        available_cars.append(car_counter)
        car_counter += 1

cars_df_sorted['car_id'] = car_ids
cars_df_sorted['accum_distance'] = accum_distances

cars_df_sorted.head(10)

,Rental_state,Start,End,Revenue,Distance,duration_minute,revenue_minute,revenue_km,car_id,accum_distance
72265,GREEN,2017-01-01 00:13:10.027,2017-01-01 00:17:56.452,0.00,0,4.773750,0.000000,0.000000,1,0
69011,RED,2017-01-01 00:19:04.324,2017-01-01 00:39:20.440,4.57,4,20.268600,0.225472,1.142500,1,4
30272,GREEN,2017-01-01 00:20:57.892,2017-01-01 02:48:11.075,0.00,0,147.219717,0.000000,0.000000,2,0
78025,RED,2017-01-01 00:21:40.974,2017-01-01 00:45:19.135,4.84,19,23.636017,0.204772,0.254737,3,19
8757,GREEN,2017-01-01 00:37:10.018,2017-01-01 00:41:21.268,0.00,0,4.187500,0.000000,0.000000,4,0
61048,RED,2017-01-01 00:37:11.831,2017-01-01 01:18:03.889,10.33,16,40.867633,0.252767,0.645625,5,16
76277,RED,2017-01-01 00:37:52.707,2017-01-01 00:50:03.954,2.42,7,12.187450,0.198565,0.345714,6,7
47985,RED,2017-01-01 00:41:27.955,2017-01-01 00:42:43.816,0.00,0,1.264350,0.000000,0.000000,1,4
83252,RED,2017-01-01 00:45:17.930,2017-01-01 01:06:59.664,4.44,7,21.695567,0.204650,0.634286,1,11
39944,RED,2017-01-01 00:47:59.854,2017-01-01 01:00:55.013,2.62,4,12.919317,0.202797,0.655000,3,23


In [8]:
cars_df_sorted.tail(10)

,Rental_state,Start,End,Revenue,Distance,duration_minute,revenue_minute,revenue_km,car_id,accum_distance
24276,RED,2017-12-31 22:18:07.545,2017-12-31 22:30:38.169,2.84,3,12.510400,0.227011,0.946667,6,154
35045,GREEN,2017-12-31 22:21:35.476,2017-12-31 23:51:26.009,0.00,0,89.842217,0.000000,0.000000,14,96
45170,RED,2017-12-31 22:30:20.209,2017-12-31 23:03:06.280,7.21,14,32.767850,0.220033,0.515000,1,59
38364,RED,2017-12-31 22:33:53.331,2017-12-31 22:43:48.866,2.18,4,9.925583,0.219634,0.545000,6,158
33763,GREEN,2017-12-31 22:40:39.742,2017-12-31 23:21:53.287,0.00,0,41.225750,0.000000,0.000000,15,156
20942,GREEN,2017-12-31 22:52:21.292,2017-12-31 23:37:50.172,0.00,0,45.481333,0.000000,0.000000,6,158
72655,RED,2017-12-31 23:03:04.342,2017-12-31 23:11:34.909,1.97,2,8.509450,0.231507,0.985000,8,72
5659,RED,2017-12-31 23:06:04.625,2017-12-31 23:20:51.704,4.17,5,14.784650,0.282049,0.834000,1,64
93123,RED,2017-12-31 23:06:37.049,2017-12-31 23:33:50.714,6.12,10,27.227750,0.224771,0.612000,9,100
32342,GREEN,2017-12-31 23:42:08.222,2017-12-31 23:46:31.453,0.00,0,4.387183,0.000000,0.000000,1,64


In [9]:
cars_df_sorted.to_csv('car_ids_fast_v2.csv', index=False)

In [ ]:
#Working version without state check
car_data = {}  # dictionary for car stats: per each car_id {'last_end': Timestamp, 'accum_distance': float, 'cooldown_until': Timestamp}
car_counter = 1
car_ids = []
accum_distances = []

for idx, row in cars_df_sorted.iterrows():
    assigned = False

    for car_id, info in car_data.items():
        #Car must be free after last rental and cooldown
        last_end = info['last_end']
        cooldown_until = info.get('cooldown_until', row['Start'])
        if row['Start'] >= max(last_end, cooldown_until):
            #Assign previous car
            car_ids.append(car_id)
            new_accum_distance = info['accum_distance'] + row['Distance']
            if new_accum_distance >= 160:
                new_accum_distance = 0
                info['cooldown_until'] = row['End'] + timedelta(minutes=540)
            info['accum_distance'] = new_accum_distance
            info['last_end'] = row['End']
            accum_distances.append(new_accum_distance)
            assigned = True
            break

    if not assigned:
        #Assign a new car
        car_ids.append(car_counter)
        accum_distance = row['Distance']
        cooldown_until = row['End'] + timedelta(minutes=540) if accum_distance >= 160 else row['Start']
        car_data[car_counter] = {
            'last_end': row['End'],
            'accum_distance': 0 if accum_distance >= 160 else accum_distance,
            'cooldown_until': cooldown_until
        }
        accum_distances.append(0 if accum_distance >= 160 else accum_distance)
        car_counter += 1

cars_df_sorted['car_id'] = car_ids
cars_df_sorted['accum_distance'] = accum_distances
cars_df_sorted.to_csv('car_ids.csv', index=False)

In [10]:
#According to my logic and assumptions the amount of cars in the fleet is 29926
print(cars_df_sorted['car_id'].nunique())


29926


I assume that the fleet size is this large because when you investigate the data you can see that there are cars available with no distance driven meaning there was no rental. I assume that this means that the customer put the car up to rent but it was never chosen. Based on the availability of the information, it is difficult to know whether it's the same car being idle over and over. However, for the purpose of this exercise it is not wholly relevant. Only the revenue generating vehicles are relevant. 

### Simulate Car battery life over 3 months for car_id = 1

In [11]:
car_id_1 = cars_df_sorted[
    (cars_df_sorted['car_id'] == 1) &
    (cars_df_sorted['End'] < pd.to_datetime('2017-03-31 23:59:59.999999'))
].copy()

#Calculation of battery left
car_id_1['battery_left'] = 100 - ((car_id_1["accum_distance"] / 200) * 100)
car_id_1.to_csv('car_id_1.csv', index=False)
#car_id_1

In [ ]:
fig = px.area(car_id_1, x="End", y="battery_left", title='Battery Level Over Time for Car ID 1 for 3 Months', 
              labels={'End': 'Time', 'battery_left': 'Battery Level (%)'})

fig.add_hline(
    y=20,
    line=dict(color='red', width=2, dash='dash'),
    annotation_text="20% minimum",
    annotation_position="bottom right"
)

fig.show()

### Incentive Program Suggestion

In [13]:
cars_master = cars_df_sorted.copy().reset_index()

#Calculation of battery left
cars_master['battery_left'] = 100 - ((cars_master["accum_distance"] / 200) * 100)
cars_master_red = cars_master[(cars_master["Rental_state"]== 'RED')]
cars_master_red = cars_master_red.drop(columns=["index"])
cars_master_red
#cars_master_red.to_csv('cars_master_red.csv', index=False)
#Filter out GREEN states as we are interested in occupied cars, which we assume are active users(customers)

,Rental_state,Start,End,Revenue,Distance,duration_minute,revenue_minute,revenue_km,car_id,accum_distance,battery_left
1,RED,2017-01-01 00:19:04.324,2017-01-01 00:39:20.440,4.57,4,20.268600,0.225472,1.142500,1,4,98.0
3,RED,2017-01-01 00:21:40.974,2017-01-01 00:45:19.135,4.84,19,23.636017,0.204772,0.254737,3,19,90.5
5,RED,2017-01-01 00:37:11.831,2017-01-01 01:18:03.889,10.33,16,40.867633,0.252767,0.645625,5,16,92.0
6,RED,2017-01-01 00:37:52.707,2017-01-01 00:50:03.954,2.42,7,12.187450,0.198565,0.345714,6,7,96.5
7,RED,2017-01-01 00:41:27.955,2017-01-01 00:42:43.816,0.00,0,1.264350,0.000000,0.000000,1,4,98.0
...,...,...,...,...,...,...,...,...,...,...,...
99992,RED,2017-12-31 22:30:20.209,2017-12-31 23:03:06.280,7.21,14,32.767850,0.220033,0.515000,1,59,70.5
99993,RED,2017-12-31 22:33:53.331,2017-12-31 22:43:48.866,2.18,4,9.925583,0.219634,0.545000,6,158,21.0
99996,RED,2017-12-31 23:03:04.342,2017-12-31 23:11:34.909,1.97,2,8.509450,0.231507,0.985000,8,72,64.0
99997,RED,2017-12-31 23:06:04.625,2017-12-31 23:20:51.704,4.17,5,14.784650,0.282049,0.834000,1,64,68.0


In [14]:
#Relationship between distance and revenue

x = cars_master_red['Distance']
y = cars_master_red['revenue_minute']

res = stats.pearsonr(x, y)
res

PearsonRResult(statistic=np.float64(-0.083096404449043), pvalue=np.float64(2.9979784611501252e-64))

The revenue per minute and distance have a very weak negative correlation of -0.08, which indicates that as the distance increases, the revenue per minute tends to decrease slightly. However, the correlation is weak, suggesting that other factors may also be influencing the revenue per minute.

In [15]:
charge_prob["Incentive_Cost"] = charge_prob["Incentive"] * 0.29
charge_prob["Incentive_Savings/Loss"] = 6 - charge_prob["Incentive_Cost"]
charge_prob

,Incentive,Charging_probability,Probability_Percent,Incentive_Cost,Incentive_Savings/Loss
0,0,0,0.00,0.00,6.00
1,5,8,0.08,1.45,4.55
2,10,16,0.16,2.90,3.10
3,15,24,0.24,4.35,1.65
4,20,32,0.32,5.80,0.20
5,25,40,0.40,7.25,-1.25
6,30,48,0.48,8.70,-2.70
7,35,56,0.56,10.15,-4.15
8,40,64,0.64,11.60,-5.60
9,45,72,0.72,13.05,-7.05


Incentive is only profitable up to 20 minutes free, i.e. when Incentive cost is less than 6€ service team charge. 


### New Features

In [ ]:


incentives_df = cars_master_red.copy()
incentives_df = incentives_df[incentives_df["Revenue"] != 0]

#Ensure data is in chronological order per car_id
incentives_df = incentives_df.sort_values(by=["car_id", "Start"])

#Reset count when accum_distance decreases compared to previous row (per car_id) to ensure accumulation resets after 160km
incentives_df["reset_flag"] = (
    incentives_df.groupby("car_id")["accum_distance"].diff() < 0
).astype(int)

incentives_df["segment"] = (
    incentives_df.groupby("car_id")["reset_flag"].cumsum() 
    + incentives_df["car_id"] * 10**6
)

incentives_df["running_duration"] = incentives_df.groupby("segment")["duration_minute"].cumsum()
incentives_df["running_revenue"] = incentives_df.groupby("segment")["Revenue"].cumsum()

#Minutes after breakeven (profitable minutes)
incentives_df["profitable_minutes"] = incentives_df["running_duration"] - 540

#Only keep positive values, as negative means not yet profitable
incentives_df["profitable_minutes"] = incentives_df["profitable_minutes"].clip(lower=0)
incentives_df["Incentive20"] = incentives_df["profitable_minutes"] - 20
incentives_df["Incentive15"] = incentives_df["profitable_minutes"] - 15
incentives_df["Incentive10"] = incentives_df["profitable_minutes"] - 10
incentives_df["Incentive5"] = incentives_df["profitable_minutes"] - 5

incentives_df

,Rental_state,Start,End,Revenue,Distance,duration_minute,revenue_minute,revenue_km,car_id,accum_distance,battery_left,reset_flag,segment,running_duration,running_revenue,profitable_minutes,Incentive20,Incentive15,Incentive10,Incentive5
1,RED,2017-01-01 00:19:04.324,2017-01-01 00:39:20.440,4.57,4,20.268600,0.225472,1.142500,1,4,98.0,0,1000000,20.268600,4.57,0.0,-20.0,-15.0,-10.0,-5.0
8,RED,2017-01-01 00:45:17.930,2017-01-01 01:06:59.664,4.44,7,21.695567,0.204650,0.634286,1,11,94.5,0,1000000,41.964167,9.01,0.0,-20.0,-15.0,-10.0,-5.0
27,RED,2017-01-01 02:01:34.286,2017-01-01 02:14:07.911,2.62,6,12.560417,0.208592,0.436667,1,17,91.5,0,1000000,54.524583,11.63,0.0,-20.0,-15.0,-10.0,-5.0
108,RED,2017-01-01 15:00:22.707,2017-01-01 15:09:30.216,1.82,1,9.125150,0.199449,1.820000,1,18,91.0,0,1000000,63.649733,13.45,0.0,-20.0,-15.0,-10.0,-5.0
114,RED,2017-01-01 15:31:38.017,2017-01-01 15:40:57.673,2.57,2,9.327600,0.275526,1.285000,1,20,90.0,0,1000000,72.977333,16.02,0.0,-20.0,-15.0,-10.0,-5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61563,RED,2017-08-17 00:08:05.554,2017-08-17 01:52:52.556,21.81,30,104.783367,0.208144,0.727000,29,44,78.0,0,29000000,138.376583,28.67,0.0,-20.0,-15.0,-10.0,-5.0
60813,RED,2017-08-15 17:34:00.629,2017-08-15 18:28:50.658,14.28,29,54.833817,0.260423,0.492414,30,29,85.5,0,30000000,54.833817,14.28,0.0,-20.0,-15.0,-10.0,-5.0
60815,RED,2017-08-15 17:36:49.228,2017-08-15 18:44:23.684,12.20,10,67.574267,0.180542,1.220000,31,10,95.0,0,31000000,67.574267,12.20,0.0,-20.0,-15.0,-10.0,-5.0
60816,RED,2017-08-15 17:37:09.225,2017-08-15 18:25:49.472,9.10,13,48.670783,0.186970,0.700000,32,13,93.5,0,32000000,48.670783,9.10,0.0,-20.0,-15.0,-10.0,-5.0


In [ ]:
car_id_1_incentive = incentives_df[(incentives_df['car_id'] == 1)]
car_id_1_incentive
car_id_1_incentive.to_csv('car_id_1_incentive.csv', index=False)

### Decision Criteria

In [30]:
profitable_rides = incentives_df[
    (incentives_df['profitable_minutes'] > 0)].drop(columns=["reset_flag", "segment", "running_duration", "running_revenue", "accum_distance"])

profitable_rides.shape

(1591, 15)

I have found 1591 rides where the life cycle of the battery is profitable.

In [32]:
profitable_rides.to_csv('profitable_rides.csv', index=False)